In [ ]:
pip install open_clip_torch

In [ ]:
import torch, random, numpy as np
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [ ]:
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms as transforms

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model and preprocess function

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

### Load the dataset and prepare the classnames

In [ ]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [ ]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1].replace('_', ' ') for pair in unique_pairs]

In [ ]:
len(r_class_names)

In [ ]:
r_class_names[:5]

In [ ]:
wnid_to_r_index = {wnid: i for i, wnid in enumerate(r_wnids)}

for wnid, idx in list(wnid_to_r_index.items())[:5]:
  print(f"wnid: {wnid}, idx: {idx}")

### Prepare the few shot and test data

In [ ]:
# from collections import defaultdict

# classes_to_indices = defaultdict(list)
# all_indices = []

# for idx in range(len(ds['test'])):
#   label = ds['test'][idx]['class_name'].replace('_', ' ')
#   classes_to_indices[label].append(idx)
#   all_indices.append(idx)

In [ ]:
from collections import defaultdict

# Pull the entire column at once (fast, single Arrow read)
class_names = ds['test']['class_name']  # list of strings

# Vectorized: build the mapping with numpy
labels = np.array([c.replace('_', ' ') for c in class_names])
unique_labels, inverse = np.unique(labels, return_inverse=True)

classes_to_indices = {
    unique_labels[i]: np.where(inverse == i)[0].tolist()
    for i in range(len(unique_labels))
}
all_indices = list(range(len(ds['test'])))

In [ ]:
print(len(classes_to_indices))
print(len(all_indices))

In [ ]:
few_shot_indices = []

for cls, indices in classes_to_indices.items():
  sampled = random.sample(indices, 16)
  few_shot_indices.extend(sampled)

In [ ]:
len(few_shot_indices) == (200 * 16)

In [ ]:
class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, preprocess, wnid_to_index):
        self.hf_dataset = hf_dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        image = self.preprocess(example["image"].convert("RGB"))
        label = self.wnid_to_index[example["wnid"]]
        return image, label

In [ ]:
full_wrapped = HFImageDataset(ds['test'], preprocess, wnid_to_r_index)

In [ ]:
from torch.utils.data import Subset

raw_few_shot_ds = Subset(full_wrapped, few_shot_indices)
print(len(raw_few_shot_ds))

In [ ]:
eval_indices = list(set(all_indices) - set(few_shot_indices))
print(len(eval_indices))

In [ ]:
raw_eval_ds = Subset(full_wrapped, eval_indices)
print(len(raw_eval_ds))

In [ ]:
raw_few_shot_loader = DataLoader(raw_few_shot_ds, batch_size = 32, shuffle = True)
raw_eval_loader = DataLoader(raw_eval_ds, batch_size = 32)

### Build the cache model for Tip-Adapter

In [ ]:
mkdir -p features

In [ ]:
from clip_zeroshot import build_and_cache_image_features, build_and_cache_text_features

In [ ]:
#few_shot_image_cache = build_and_cache_image_features(model, device, raw_few_shot_loader, './features', 'few_shot_image_features')

In [ ]:
# few_shot_image_features = few_shot_image_cache['image_features']
# few_shot_image_labels = few_shot_image_cache['labels']

In [ ]:
# CAUTION: Only use if to load cached features

from clip_zeroshot import load_cached_image_features

loaded_few_shot_image_cache = load_cached_image_features('/content/features/r_few_shot_image_features.pt')

few_shot_image_features = loaded_few_shot_image_cache['image_features']
few_shot_image_labels = loaded_few_shot_image_cache['labels']

In [ ]:
print(few_shot_image_features.shape)
print(few_shot_image_labels.shape)

In [ ]:
few_shot_image_labels[:10]

In [ ]:
import torch.nn.functional as F

one_hot = F.one_hot(few_shot_image_labels, num_classes=len(r_class_names))

In [ ]:
print(one_hot.shape)

In [ ]:
cache_keys = few_shot_image_features
cache_values = one_hot.float()

### Build the zero shot classifier

In [ ]:
from imagenet_classes import IMAGENET_TEMPLATES

In [ ]:
#text_features = build_and_cache_text_features(model = model, device= device, tokenizer = tokenizer, classnames = r_class_names, templates = IMAGENET_TEMPLATES, cache_dir='./features', file_name='text-features')

In [ ]:
from clip_zeroshot import load_cached_text_features

text_feature_cache = load_cached_text_features('/content/features/r_text-features.pt')
text_features = text_feature_cache['text_features']

### Build the test features

In [ ]:
# eval_cache = build_and_cache_image_features(model, device, raw_eval_loader, './features', 'r_eval_features')
# eval_features = eval_cache['image_features']
# eval_labels = eval_cache['labels']

In [ ]:
# CAUTION: Only use it to load cached features

from clip_zeroshot import load_cached_image_features

load_eval_cached = load_cached_image_features('/content/features/r_eval_features.pt')
eval_features = load_eval_cached['image_features']
eval_labels = load_eval_cached['labels']

### Build the coop's text features

In [ ]:
from coop import PromptLearner, TextEncoderWrapper

In [ ]:
text_encoder = TextEncoderWrapper(model)
coop_prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=r_class_names).to(device)
coop_prompt_learner.ctx.data.copy_(torch.load('/content/features/coop_ctx_200_cls.pt'))

with torch.no_grad():
    prompts, tok = coop_prompt_learner()
    coop_text_features = text_encoder(prompts, tok)
    coop_text_features = coop_text_features / coop_text_features.norm(dim=-1, keepdim=True)

### Setup TPT inputs

In [ ]:
augment_transform = transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)) # from clip
])

In [ ]:
# TPT needs raw images + labels (a subset), not cached features
tpt_images = [ds['test'][i]['image'] for i in range(200)]
tpt_labels = [wnid_to_r_index[ds['test'][i]['wnid']] for i in range(200)]

# a fresh, resettable prompt learner (TPT's, with R class names)
tpt_prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=r_class_names).to(device)

### Run the harness

In [ ]:
if 'harness' in sys.modules:
    del sys.modules['harness']

In [ ]:
from harness import run_comparison, zero_shot_logits, tip_adapter_logits, coop_logits, run_tpt, accuracy, ece

In [ ]:
metrics = {"accuracy": accuracy, "ece": ece}

In [ ]:
shared = {
    "test_features": eval_features.to(device),
    "labels": eval_labels.to(device),
    "text_features": text_features.to(device),
    "cache_keys": cache_keys.to(device),
    "cache_values": cache_values.to(device),
    "logit_scale": model.logit_scale.exp(),
    "coop_text_features": coop_text_features.to(device)
}

In [ ]:
methods = {
    "zero_shot":   {"fn": zero_shot_logits,   "params": {}},
    "tip_adapter": {"fn": tip_adapter_logits, "params": {"alpha": 1.5, "beta": 5.0}},
    "coop":        {"fn": coop_logits,        "params": {}}
}

In [ ]:
results = run_comparison(shared, methods, metrics)

In [ ]:
for p in model.parameters():
    p.requires_grad_(False)
print(sum(p.requires_grad for p in model.parameters()))   # must be 0

In [ ]:
tpt_result = run_tpt(model, tpt_prompt_learner, text_encoder, preprocess, tpt_images, tpt_labels, device, augment_transform, metrics)
results["tpt"] = tpt_result

In [ ]:
print(results)

In [ ]:
for method, r in results.items():
  n = r.get("n", "full")
  print(f"{method:12s}: {r['accuracy']:.2f}%  {r['ece']:.2f}% (n={n})")